# Neural feature harmonization

Twelve datasets in this corpus ship **pre-binned** neural features, and none of them agree on
what a number means. This notebook shows what `NeuralFeatureExtractor` does about that, and
checks it against the real corpus rather than against prose.

It is the executable reference for the `units and normalization` section of
[`docs/standardization.md`](../standardization.md). Every claim there is asserted here, so if
the code and the documentation drift apart, this notebook fails.

**Contents**

1. The problem — same neuron, different numbers
2. The fields every event declares
3. What the corpus actually declares
4. Conversion, and why it runs before resampling
5. What must never be converted
6. The normalization chain
7. End to end: two Studies through one pipeline
8. Next: `NormalizedExtractor`

In [1]:
import ast
import logging
import pathlib
import tempfile

import neuralset as ns
import numpy as np

import neuralshed
from neuralshed import units
from neuralshed.datasets.utils import dataset_dir
from neuralshed.datasets.willet_2021.study import Willett2021Handwriting
from neuralshed.datasets.wilson_2025.study import Wilson2025CursorRecalibration
from neuralshed.events import BandPower, SpikeCounts
from neuralshed.extractors import NeuralFeatureExtractor

logging.getLogger("neuralset").setLevel(logging.WARNING)  # keep the pipeline output readable

## 1. The problem

A channel firing at a steady **20 Hz** is a physical fact. Here is how three datasets in this
corpus store it.

In [2]:
def as_stored(rate_hz, bin_ms, unit):
    """What a channel firing at `rate_hz` looks like in one dataset's storage convention."""
    return rate_hz if unit == "Hz" else rate_hz * bin_ms / 1000


for name, bin_ms, unit in [
    ("singer_2025", 10, "counts/bin"),
    ("wilson_2025", 20, "Hz"),
    ("deo_2024", 20, "counts/bin"),
]:
    print(f"{name:14s} {bin_ms:>3} ms bins  {unit:11s} -> stores {as_stored(20.0, bin_ms, unit)}")

singer_2025     10 ms bins  counts/bin  -> stores 0.2
wilson_2025     20 ms bins  Hz          -> stores 20.0
deo_2024        20 ms bins  counts/bin  -> stores 0.4


`0.2`, `20.0` and `0.4` for the same neuron. A model pooling these sees a factor of 100 between
the first two — and a factor of 2 between the two that *share* a unit string.

That second pair is the point worth dwelling on. Agreeing on the unit is not enough, because
**`counts/bin` is not self-contained**: it is a count per *unspecified* bin width, and it only
means something paired with `frequency`. `Hz` is self-contained, which is why it is the
harmonization target.

## 2. The fields every event declares

`DenseArrayEvent` carries `unit`, `normalization` and `processing`. They look similar and are
not.

| field | question it answers | who reads it |
|---|---|---|
| `unit` | *how do I convert this?* | `NeuralFeatureExtractor`, with `frequency` |
| `normalization` | *which statistic was removed?* | the pooling decision, and the rule in §5 |
| `processing` | *what else did the source do?* | people, and pooling code comparing two streams at the same rank |

Keeping them apart is what lets `unit` record the **pre-normalization** quantity. `card_2024`
ships z-scored threshold crossings and declares `counts/bin`, because that is what the numbers
were before the authors z-scored them. A single string had to call that `"z-scored (a.u.)"` and
threw the quantity away.

In [3]:
print("unit classes")
for u in ["counts/bin", "Hz", "uV^2", "a.u.", "px"]:
    print(f"  {u:12s} {units.classify(u)}")

print()
print("normalization, least to most processed")
print(" ", " -> ".join(units.NORMALIZATION_ORDER))

unit classes
  counts/bin   count
  Hz           rate
  uV^2         non_convertible
  a.u.         non_convertible
  px           non_convertible

normalization, least to most processed
  raw -> mean_subtracted -> z_scored


`count` is **extensive**: it scales with bin width. `rate` is **intensive**: it does not. That
distinction is the entire mechanism, and it is not our invention — `jude_2026_speech`'s own authors
*sum* threshold crossings and *average* spike-band power when they compress bins, which is
exactly why crossings are `counts/bin` and power is `uV^2`.

## 3. What the corpus actually declares

Rather than trust a table someone typed, read the declarations straight out of the Study
sources. `tests/test_units.py` runs the same scan in CI, so a dataset introducing an
unregistered string fails the build rather than someone's training run.

In [4]:
NEURAL = {"SpikeCounts", "BandPower"}


def _lit(node):
    """String literals a node can evaluate to, expanding a conditional into both branches."""
    if isinstance(node, ast.Constant) and isinstance(node.value, str):
        return {node.value}
    if isinstance(node, ast.IfExp):
        return _lit(node.body) | _lit(node.orelse)
    return set()


def neural_rows(path):
    """(units, normalizations) from every call that declares a neural `type`.

    Per row, not per module: most studies emit behavioral series from the same function, and
    their `px` / `binary` units are not part of this table.
    """
    tree = ast.parse(path.read_text())
    helpers = {}
    for fn in ast.walk(tree):
        if not isinstance(fn, ast.FunctionDef):
            continue
        names = [a.arg for a in fn.args.args]
        positional = {
            a: names.index(a)
            for a in ("type_", "type", "unit", "normalization")
            if a in names
        }
        # a helper may set a field in its own body rather than take it as an argument
        in_body = {
            a: {
                v
                for n in ast.walk(fn)
                if isinstance(n, ast.Call)
                for kw in n.keywords
                if kw.arg == a
                for v in _lit(kw.value)
            }
            for a in ("unit", "normalization")
        }
        helpers[fn.name] = (positional, in_body)

    found = {"unit": set(), "normalization": set()}
    for node in ast.walk(tree):
        if not isinstance(node, ast.Call):
            continue
        name = getattr(node.func, "id", None) or getattr(node.func, "attr", None)
        positional, in_body = helpers.get(name, ({}, {}))
        kw = {k.arg: k.value for k in node.keywords}

        types = _lit(kw["type"]) if "type" in kw else set()
        for key in ("type_", "type"):
            if key in positional and positional[key] < len(node.args):
                types |= _lit(node.args[positional[key]])
        if not types & NEURAL:
            continue

        for arg in ("unit", "normalization"):
            got = _lit(kw[arg]) if arg in kw else set()
            if not got and arg in positional and positional[arg] < len(node.args):
                got = _lit(node.args[positional[arg]])
            found[arg] |= got or in_body.get(arg, set())
    return found["unit"], found["normalization"]


root = pathlib.Path(neuralshed.__file__).parent / "datasets"
print(f"{'dataset':16s} {'neural unit(s)':22s} normalization")
for study_py in sorted(root.glob("*/study.py")):
    declared_units, declared_norms = neural_rows(study_py)
    declared_units = {units.base_unit(u) for u in declared_units if units.is_registered(u)}
    declared_norms = declared_norms & units.NORMALIZATIONS
    if declared_units:  # fan_h1 emits spike times, not a dense array, so it drops out
        print(
            f"{study_py.parent.name:16s} {', '.join(sorted(declared_units)):22s} "
            f"{', '.join(sorted(declared_norms))}"
        )

dataset          neural unit(s)         normalization
fan_2023           counts/bin             raw
wairagkar_2025   counts/bin, uV^2       z_scored
willet_2021      counts/bin             raw
willet_2023      counts/bin, uV^2       raw


In [5]:
unregistered = {
    (p.parent.name, u)
    for p in sorted(root.glob("*/study.py"))
    for u in neural_rows(p)[0]
    if not units.is_registered(u)
}
assert not unregistered, unregistered
print("every declared neural unit is registered")

every declared neural unit is registered


Three rows are worth reading closely.

`wilson_2025` is the only dataset already in `Hz` — its source scales each bin count to a rate.
`card_2024` declares a physical unit *and* `z_scored`, which is the split this schema exists
for. `wairagkar_2025` is the same split taken further: its pipeline log-transforms before
normalizing, so the shipped values are dimensionless, yet `unit` still names what fed that
chain. That is the quantity another dataset would have to be converted into to join it.

## 4. Conversion, and why it runs before resampling

`target_unit="Hz"` puts every count dataset on the rate scale, using `event.unit` and both the
source and destination rates. `"native"` keeps whatever unit the source stored, and inspects
nothing at all when the rate is unchanged. Every call site states its target explicitly rather
than leaning on the field default.

The helpers below build a real event on disk and call the extractor the way anything else
does, through `__call__`.

In [6]:
workdir = pathlib.Path(tempfile.mkdtemp())
_made = 0


def make(values, unit, frequency, cls=SpikeCounts, normalization="raw"):
    """A DenseArrayEvent backed by a real .npy, so `read()` follows the normal path."""
    global _made
    _made += 1
    values = np.asarray(values, dtype=np.float32)
    path = workdir / f"feat{_made}.npy"
    np.save(path, values)
    return cls(
        timeline="demo",
        subject="s1",
        feature="threshold_crossings",
        start=0.0,
        duration=values.shape[-1] / frequency,
        frequency=frequency,
        unit=unit,
        normalization=normalization,
        filepath=str(path),
    )


def extract(event, target_unit, frequency="native"):
    """Values the extractor yields for one event, plus the unit label it attaches."""
    ex = NeuralFeatureExtractor(
        frequency=frequency,
        target_unit=target_unit,
        event_types=type(event).__name__,
    )
    values = np.asarray(ex(event, start=event.start, duration=event.duration))
    header = next(ex._get_data([event])).header  # internal view: where the label lives
    return values, header

In [7]:
# the same 20 Hz channel, stored two ways
fast = make(np.full((1, 10), 0.2), "counts/bin", 100.0)  # 10 ms bins, as counts
slow = make(np.full((1, 5), 20.0), "Hz", 50.0)  # 20 ms bins, as a rate

print("as stored :", extract(fast, "native")[0][0, 0], "vs", extract(slow, "native")[0][0, 0])
print("as Hz     :", extract(fast, "Hz")[0][0, 0], "vs", extract(slow, "Hz")[0][0, 0])

assert np.allclose(extract(fast, "Hz")[0], 20.0)
assert np.allclose(extract(slow, "Hz")[0], 20.0)

as stored : 0.2 vs 20.0
as Hz     : 20.0 vs 20.0


### The order is load-bearing

Conversion runs *before* resampling, and it needs **both** rates. An extensive quantity has to
change when the bin width changes, so a count's factor depends on the destination bin as much
as on the source one: `counts/bin` at 100 Hz reaching `counts/bin` at 50 Hz is a factor of 2,
not a no-op.

Resample a count stream from 100 Hz to 50 Hz and the honest answer is that each new 20 ms bin
holds *twice* the count. Ask for `Hz` instead and the question never arises, because a rate is
a rate at any bin width.

In [8]:
# a varying signal — a constant one would pass even against a broken resampler
counts = np.array([[0.0, 0.4, 0.1, 0.9, 0.2, 0.6, 0.3, 0.5]], dtype=np.float32)
event = make(counts, "counts/bin", 100.0)

as_hz, hz_header = extract(event, "Hz", frequency=50.0)
native, native_header = extract(event, "native", frequency=50.0)
merged = counts.reshape(1, 4, 2).sum(axis=2)  # what a 20 ms bin holds

print("source counts @100 Hz :", counts[0])
print("converted   @ 50 Hz   :", as_hz[0], f"({hz_header['unit']})")
print("implied counts        :", as_hz[0] / 50.0)
print("expected counts       :", merged[0])
print("unconverted @ 50 Hz   :", native[0], f"({native_header['unit']})")

assert np.allclose(as_hz / 50.0, merged)
# `native` keeps the source's unit string, so it has to keep the values true to it
assert np.allclose(native, merged)

source counts @100 Hz : [0.  0.4 0.1 0.9 0.2 0.6 0.3 0.5]
converted   @ 50 Hz   : [20.       50.       40.000004 40.      ] (Hz)
implied counts        : [0.4       1.        0.8000001 0.8      ]
expected counts       : [0.4 1.  0.8 0.8]
unconverted @ 50 Hz   : [0.4 1.  0.8 0.8] (counts/bin)


### How the rebinning itself works

`NeuralFeatureExtractor._resample` is **area-preserving**, not interpolation. Each destination
bin is the source averaged over that bin's own span, computed by interpolating the running
*integral* rather than the signal, which is exact because the integral of a piecewise-constant
signal is piecewise-linear.

So 100 Hz to 20 Hz merges each group of five bins instead of reading one and discarding four,
upsampling splits a bin evenly, and a non-integer ratio splits proportionally. The invariant is
that the spike total over the window is conserved in either direction.

The base class keeps `np.interp` for the behavioral streams, where a sampled continuous signal
is genuinely what interpolation assumes.

Two things rebinning cannot fix. A **z-scored** stream merges by averaging, which is not the
z-score of the wider bin — that would need statistics the source never released — so a rebinned
z-scored stream is no longer unit-variance. And **upsampling** invents no structure below the
source bin width, because none is recoverable.

## 5. What must never be converted

Two guards, for two different reasons.

**Spike-band power** is `uV^2` — neither a count nor a rate, and intensive already, so no
frequency factor means anything. Seven datasets emit it, and a blanket coerce-to-Hz would
corrupt all of them.

In [9]:
power_values = np.array([[1.0, 250.0, 31400.0]], dtype=np.float32)
power = make(power_values, "uV^2", 100.0, cls=BandPower)

for target in ("native", "Hz", "counts/bin"):
    out, header = extract(power, target)
    assert np.array_equal(out, power_values)
    print(f"target={target:11s} -> unchanged, header says {header['unit']!r}")

target=native      -> unchanged, header says 'uV^2'
target=Hz          -> unchanged, header says 'uV^2'
target=counts/bin  -> unchanged, header says 'uV^2'


**Already-normalized features** are the subtler case, and the reason `normalization` is its own
field.

`card_2024` now declares `unit="counts/bin"` — the pre-normalization quantity. That string, on
its own, *looks* convertible. What stops it is the interaction rule:

> Unit conversion is valid **iff** `normalization` is `raw` or `mean_subtracted`.

Once a per-channel standard deviation has been divided out the values are dimensionless, and
multiplying them by a frequency is meaningless. So it is a **no-op, not an error** — and the
header keeps reporting the source unit, so it never claims a conversion that did not happen.

In [10]:
values = np.array([[-1.5, 0.0, 2.25]], dtype=np.float32)

for regime in units.NORMALIZATION_ORDER:
    ev = make(values, "counts/bin", 50.0, normalization=regime)
    out, header = extract(ev, "Hz")
    scaled = not np.array_equal(out, values)
    print(
        f"normalization={regime:16s} converts={units.preserves_unit(regime)!s:5s} "
        f"scaled={scaled!s:5s} header unit={header['unit']!r}"
    )
    assert scaled == units.preserves_unit(regime)

normalization=raw              converts=True  scaled=True  header unit='Hz'
normalization=mean_subtracted  converts=True  scaled=True  header unit='Hz'
normalization=z_scored         converts=False scaled=False header unit='counts/bin'


Note the `mean_subtracted` row: it **does** convert. An additive offset commutes with
rebinning, since `(c1 - u) + (c2 - u) = (c1 + c2) - 2u` and `2u` is exactly the wider bin's own
mean, so a mean-subtracted count is still a count. Z-scoring divides by a standard deviation
that does *not* scale linearly with bin width, which is what breaks the equivalence.

## 6. The normalization chain

`raw -> mean_subtracted -> z_scored`, and never backwards. The per-channel statistics the
authors divided out live in their analysis code; they were not deposited.

```
raw  -->  mean_subtracted  -->  z_scored
 ^              ^                   |
 +--------------+------  X  --------+
```

The field names which statistic was removed. Everything else a source's
chain did goes in `event.processing`.

In [11]:
regimes = units.NORMALIZATION_ORDER
w = max(len(r) for r in regimes)
print(f"{'from / to':>{w}}   " + "  ".join(f"{r:>{w}}" for r in regimes))
for src in regimes:
    marks = "  ".join(
        f"{('yes' if units.is_reachable(src, dst) else '-'):>{w}}" for dst in regimes
    )
    print(f"{src:>{w}}   {marks}")

assert units.is_reachable("raw", "z_scored")
assert not units.is_reachable("z_scored", "raw")
assert not units.is_reachable("z_scored", "mean_subtracted")

      from / to               raw  mean_subtracted         z_scored
            raw               yes              yes              yes
mean_subtracted                 -              yes              yes
       z_scored                 -                -              yes


The consequence is arithmetic, not preference: **a pool's target must be at least as processed
as every member of it.** Include `card_2024` or `wairagkar_2025`, both z-scored, and the whole
pool is dragged up the chain.

Equal rank is not the same as interchangeable, though, and that is what `processing` records.
`card_2024` z-scores over the preceding ~20 trials. `wairagkar_2025` log-transforms first, then
z-scores over a rolling 10 s window, then smooths. Both are `z_scored`, so `is_reachable` calls
them mutually reachable, and it is wrong to trust that alone — nothing yet compares the chains.

What levelling up costs is the physical scale, which the next section shows you would be
giving up.

## 7. End to end: two Studies through one pipeline

Everything above used hand-built events. This section runs the real path — **Study -> events
-> `Segmenter` -> batch tensors** — over two datasets at once.

`willet_2021` and `wilson_2025` recorded the **same BrainGate2 participant (T5)** through the
same arrays at the same threshold. They store their features differently: 100 Hz counts versus
50 Hz rates. If the conversion is right, both should report the same firing rate.

Needs the source drive mounted; skipped cleanly otherwise.

In [12]:
STUDIES = [
    ("willet_2021", Willett2021Handwriting),
    ("wilson_2025", Wilson2025CursorRecalibration),
]

available = all((dataset_dir(name) / "data").exists() for name, _ in STUDIES)
print("source data available:", available)

events = {}
if available:
    for name, cls in STUDIES:
        study = cls(path=str(dataset_dir(name) / "data"), query="timeline_index < 1")
        events[name] = study.run()
        neural = events[name][events[name].type == "SpikeCounts"]
        print(
            f"{name:14s} {len(events[name]):>5} events, neural stream @ "
            f"{neural.frequency.iloc[0]:.0f} Hz declaring "
            f"{neural.unit.iloc[0]!r} / {neural.normalization.iloc[0]!r}"
        )

source data available: True
willet_2021      103 events, neural stream @ 100 Hz declaring 'counts/bin' / 'raw'


wilson_2025      194 events, neural stream @ 50 Hz declaring 'Hz' / 'raw'


One `Segmenter`, one extractor, two datasets. The only thing that changes between the two runs
below is `target_unit`.

In [13]:
WINDOW_S = 30.0
COMMON_HZ = 50.0


def batch(df, target_unit, frequency=COMMON_HZ, duration=WINDOW_S):
    """One `(1, n_channels, T)` batch, straight off the segmenter."""
    segmenter = ns.dataloader.Segmenter(
        extractors={
            "neural": NeuralFeatureExtractor(
                frequency=frequency, event_types="SpikeCounts", target_unit=target_unit
            )
        },
        trigger_query="type == 'SpikeCounts'",
        duration=duration,
    )
    segments = segmenter.apply(df)
    segments.prepare()
    return np.asarray(segments[0].data["neural"])


rates = {}
for name, df in events.items():
    stored, harmonized = batch(df, "native"), batch(df, "Hz")
    rates[name] = (float(stored.mean()), float(harmonized.mean()))
    print(
        f"{name:14s} batch {stored.shape}   target_unit='native' {rates[name][0]:9.4f}   "
        f"target_unit='Hz' {rates[name][1]:7.3f}"
    )

willet_2021    batch (1, 192, 1500)   target_unit='native'    0.4325   target_unit='Hz'  21.627
wilson_2025    batch (1, 192, 1500)   target_unit='native'   24.6854   target_unit='Hz'  24.685


In [14]:
if rates:
    stored = [v[0] for v in rates.values()]
    harmonized = [v[1] for v in rates.values()]
    print(
        f"as stored  : {stored[0]:.4f} vs {stored[1]:.4f}"
        f"   -> {max(stored) / min(stored):.0f}x apart"
    )
    print(
        f"harmonized : {harmonized[0]:.2f} Hz vs {harmonized[1]:.2f} Hz"
        f"   -> {max(harmonized) / min(harmonized):.2f}x apart"
    )

    # same participant, same arrays, same threshold: the rates must be close
    assert max(harmonized) / min(harmonized) < 1.5
    print()
    print("Two datasets, one participant, one scale — and it only exists in Hz.")
    print("Levelling a pool to z_scored would discard this check.")

as stored  : 0.4325 vs 24.6854   -> 57x apart
harmonized : 21.63 Hz vs 24.69 Hz   -> 1.14x apart

Two datasets, one participant, one scale — and it only exists in Hz.
Levelling a pool to z_scored would discard this check.


### Rebinning on real data

Section 4 showed this on eight hand-made numbers. Here it is on a real block.

`willet_2021` bins at 100 Hz. Ask for its **native counts** at 50 Hz and each 20 ms bin now
holds what its two 10 ms bins held, so the per-bin value doubles. Ask for `Hz` and the answer
is stable across both rates, because that is what intensive means.

In [15]:
if events:
    df = events["willet_2021"]
    at_source_rate = batch(df, "native", frequency=100.0).mean()
    rebinned = batch(df, "native", frequency=COMMON_HZ).mean()

    print("target_unit='native' (counts/bin)")
    print(f"  @100 Hz, the source rate : {at_source_rate:.4f}")
    print(f"  @ 50 Hz, rebinned        : {rebinned:.4f}   <- a 20 ms bin holds twice as much")
    print()
    print("target_unit='Hz'")
    print(f"  @100 Hz : {batch(df, 'Hz', frequency=100.0).mean():.3f}")
    print(f"  @ 50 Hz : {batch(df, 'Hz', frequency=COMMON_HZ).mean():.3f}   <- invariant")

    assert abs(rebinned / at_source_rate - 2.0) < 0.1

target_unit='native' (counts/bin)
  @100 Hz, the source rate : 0.2163
  @ 50 Hz, rebinned        : 0.4325   <- a 20 ms bin holds twice as much

target_unit='Hz'


  @100 Hz : 21.627


  @ 50 Hz : 21.627   <- invariant


## 8. Next: `NormalizedExtractor`

Everything above harmonizes the **unit** and the **rate**. It deliberately does not touch the
**normalization** — `NeuralFeatureExtractor` reads that field and never changes it, which is
what keeps it stateless and cacheable per event.

Moving a dataset up the lattice needs fitted per-channel statistics, and that is a different
kind of object: one with a `prepare` step, a fitting set, and every leakage trap that comes
with them. It arrives as `NormalizedExtractor`, and this notebook gains a section then.

Until that lands, a pool is limited to datasets that already share a normalization regime —
which, per the table in section 3, is most of the corpus.